# Execution Tactics

This notebook accompanies the **Execution Tactics** chapter. It implements:

1. **Fill probability model** — exponential fill-rate calibration.
2. **Single-venue optimal tactic** — HJB backward induction and aggressiveness matrix.
3. **Smart order routing (SOR)** — multi-venue benefit via aggregated fill rates.
4. **Reinforcement learning** — tabular Q-learning for adaptive execution.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
FIGURES_DIR = '../markdown/figures'
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12})

## 1  Fill probability model

Orders posted at depth $\delta$ ticks away from the best quote are filled at a Poisson rate

$$
\lambda(\delta) = A\,e^{-k\delta},
$$

where $A$ is the baseline arrival rate and $k$ controls how fast the fill rate decays with depth.
The probability of at least one fill within a window of duration $\tau$ is

$$
P(\text{fill}\mid\delta,\tau) = 1 - e^{-\lambda(\delta)\,\tau}.
$$

Calibration in practice uses regression of historical fill times against posted depth; the
exponential form is justified by the empirical depth distribution of the limit order book
(Cont, Stoikov & Talreja 2010).


In [ ]:
def fill_rate(delta, A, k):
    return A * np.exp(-k * delta)

def fill_prob(delta, tau, A, k):
    return 1 - np.exp(-fill_rate(delta, A, k) * tau)

A_base = 1.0
k_vals = [0.5, 1.0, 2.0]
deltas = np.linspace(0, 6, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
clrs = ['#1565C0', '#2E7D32', '#C62828']

for k, col in zip(k_vals, clrs):
    axes[0].plot(deltas, fill_rate(deltas, A_base, k), lw=2.2, color=col, label=f'$k = {k}$')
axes[0].set_xlabel('Depth $\\delta$ (ticks)')
axes[0].set_ylabel('Fill rate $\\lambda(\\delta)$  (fills/min)')
axes[0].set_title('Fill rate vs placement depth')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

tau_vals = np.linspace(0, 30, 300)
depth_ex = [0, 2, 4]
d_clrs   = ['#9C27B0', '#FF6F00', '#00796B']
for d, col in zip(depth_ex, d_clrs):
    axes[1].plot(tau_vals, fill_prob(d, tau_vals, A_base, 1.0),
                 lw=2.2, color=col, label=f'$\\delta = {d}$ ticks')
axes[1].set_xlabel('Time window $\\tau$ (min)')
axes[1].set_ylabel('$P(\\mathrm{fill}\\mid\\delta,\\tau)$')
axes[1].set_title('Fill probability vs window  ($A=1$, $k=1$)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(f'{FIGURES_DIR}/tact_fill_probability.png', dpi=150, bbox_inches='tight')
plt.show()

## 2  Single-venue optimal tactic

### 2.1  Hamilton–Jacobi–Bellman equation

Let $H(t,q)$ be the maximum expected cash proceeds from selling $q$ remaining units with
$T-t$ time left. The HJB equation is

$$
-\partial_t H = \max_{\delta \ge 0}\;\bigl\{\lambda(\delta)\,[\delta + H(t,q-1) - H(t,q)]\bigr\}
$$

with terminal condition $H(T,q) = -b\,q$ (forced market liquidation at cost $b$ per unit).

Setting the derivative with respect to $\delta$ to zero gives

$$
\delta^*(t,q) = \frac{1}{k} + \Delta H(t,q), \qquad \Delta H = H(t,q) - H(t,q-1) \le 0.
$$

Substituting back yields the reduced ODE in remaining time $\tau = T - t$:

$$
\frac{dH}{d\tau} = \frac{A}{e\,k}\,\exp\!\bigl(-k\,\Delta H\bigr).
$$

### 2.2  Backward induction

We integrate the ODE forward in $\tau$ using Euler's method, starting from the terminal
condition $H(\tau=0,q) = -b\,q$.


In [ ]:
def compute_hjb(A_list, k_list, b, T, Q_max=20, N_tau=600):
    """
    Multi-venue HJB backward induction.

    Returns
    -------
    H_arr     : (N_tau+1, Q_max+1) value function, index 0 = terminal
    delta_arr : (N_tau+1, Q_max+1, n_venues) optimal depths
    tau_grid  : (N_tau+1,)
    """
    dtau = T / N_tau
    H = np.array([-b * q for q in range(Q_max + 1)], dtype=float)
    H_arr    = np.zeros((N_tau + 1, Q_max + 1))
    delta_arr = np.zeros((N_tau + 1, Q_max + 1, len(A_list)))
    H_arr[0] = H.copy()

    for step in range(N_tau):
        H_new = np.zeros(Q_max + 1)
        H_new[0] = 0.0
        for q in range(1, Q_max + 1):
            DH  = np.clip(H[q] - H[q - 1], -30.0, 5.0)
            rhs = sum(A * np.exp(-k * DH) / (np.e * k)
                      for A, k in zip(A_list, k_list))
            H_new[q] = H[q] + dtau * rhs
            for v, k_v in enumerate(k_list):
                delta_arr[step + 1, q, v] = 1.0 / k_v + DH
        H = H_new.copy()
        H_arr[step + 1] = H.copy()

    return H_arr, delta_arr, np.linspace(0, T, N_tau + 1)

In [ ]:
# ── Aggressiveness matrices for three urgency levels ──────────────────────
A_mat, k_mat, T_mat, Q_max = 1.0, 1.0, 10.0, 20
b_vals = [0.5, 2.0, 5.0]
titles = ['Low urgency  ($b = 0.5$)', 'Moderate  ($b = 2$)', 'High urgency  ($b = 5$)']

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, b, title in zip(axes, b_vals, titles):
    H_arr, d_arr, _ = compute_hjb([A_mat], [k_mat], b, T_mat, Q_max=Q_max, N_tau=600)
    # flip tau: column 0 = execution start, column -1 = terminal deadline
    D_plot = np.flip(d_arr[1:, 1:, 0].T, axis=1)
    vmin, vmax = -0.5, min(float(D_plot.max()), 7.0)
    im = ax.imshow(D_plot, aspect='auto', origin='lower',
                   extent=[0, 1, 1/Q_max, 1], cmap='RdYlGn_r', vmin=vmin, vmax=vmax)
    t_r = np.linspace(0, 1, D_plot.shape[1])
    q_r = np.linspace(1/Q_max, 1, Q_max)
    Tg, Qg = np.meshgrid(t_r, q_r)
    try:
        ax.contour(Tg, Qg, D_plot, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
    except Exception:
        pass
    ax.set_xlabel('Time elapsed $t/T$')
    if ax is axes[0]:
        ax.set_ylabel('Remaining fraction $q/Q$')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='$\\delta^*$ (ticks)')

fig.suptitle('Optimal aggressiveness matrices  (dashed = market-order boundary)', fontsize=12, y=1.02)
plt.tight_layout()
fig.savefig(f'{FIGURES_DIR}/tact_aggressiveness_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('Aggressiveness matrix at start (tau=T, q=5..20, b=2):')
_, d_arr2, _ = compute_hjb([A_mat], [k_mat], 2.0, T_mat, Q_max=Q_max, N_tau=600)
for q in [1, 5, 10, 15, 20]:
    print(f'  q={q:2d}: delta* = {d_arr2[-1, q, 0]:.3f}')

### 2.3  Simulation of the single-venue tactic

We simulate executions by reading $\delta^*$ from the lookup table at each step, drawing
Poisson inter-fill times, and applying the terminal liquidation penalty $b$ to any unfilled units.


In [ ]:
def simulate_tactic(H_arr, delta_arr_v0, A, k, b, Q, T, n_sims=5000, seed=0):
    """
    Single-venue optimal tactic simulation.
    delta* < 0 triggers an intra-window market order (proceeds = 0).
    Residual at T costs b per unit.
    """
    N_tau = H_arr.shape[0] - 1
    rng   = np.random.default_rng(seed)
    proceeds = np.zeros(n_sims)
    for s in range(n_sims):
        q, tau, total = Q, T, 0.0
        while q > 0 and tau > 1e-9:
            tau_idx = int(np.clip(tau / T * N_tau, 0, N_tau))
            q_idx   = min(q, delta_arr_v0.shape[1] - 1)
            d       = float(delta_arr_v0[tau_idx, q_idx, 0])
            if d < 0:
                q -= 1       # market order at mid, no price improvement
                continue
            lam  = A * np.exp(-k * d)
            wait = rng.exponential(1.0 / lam) if lam > 1e-12 else np.inf
            if wait < tau:
                tau -= wait; total += d; q -= 1
            else:
                tau = 0
        if q > 0:
            total -= b * q
        proceeds[s] = total
    return proceeds

# Compare three inventory levels (b=2.0)
A_sim, k_sim, b_sim, T_sim = 1.0, 1.0, 2.0, 10.0
H_sim, d_sim, _ = compute_hjb([A_sim], [k_sim], b_sim, T_sim, Q_max=20, N_tau=600)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, Q_test in zip(axes, [5, 10, 20]):
    proc = simulate_tactic(H_sim, d_sim, A_sim, k_sim, b_sim, Q_test, T_sim, n_sims=5000)
    ax.hist(proc, bins=40, color='#1565C0', alpha=0.7, density=True)
    ax.axvline(proc.mean(), color='#C62828', lw=2, linestyle='--',
               label=f'mean = {proc.mean():.2f}')
    ax.set_title(f'$Q = {Q_test}$ units')
    ax.set_xlabel('Total proceeds (ticks)')
    ax.set_ylabel('Density')
    ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('Proceeds distribution — single-venue optimal tactic  ($b = 2$)', fontsize=12)
plt.tight_layout()
plt.show()

## 3  Smart Order Routing (SOR)

When orders can be posted simultaneously on $K$ venues, the total fill rate is

$$
\lambda_{\mathrm{tot}}(\boldsymbol{\delta}) = \sum_{k=1}^{K} A_k\,e^{-k_k\,\delta^k}.
$$

Each venue contributes independently, so the combined HJB simply sums per-venue contributions:

$$
\frac{dH}{d\tau} = \sum_{k=1}^{K} \frac{A_k}{e\,k_k}\,\exp(-k_k\,\Delta H),
$$

and the optimal depth for venue $k$ is $\delta^{k,*} = 1/k_k + \Delta H$ (same $\Delta H$ for all venues).

### Key SOR insight

Adding a second venue increases $\lambda_{\mathrm{tot}}$, reducing expected waiting time for the
next fill. This lowers the probability of hitting the terminal deadline with residual inventory,
improving execution quality even when the second venue is illiquid.


In [ ]:
def simulate_sor_fixed(A_list, k_list, delta_fixed, b, Q, T, n_sims=10_000, seed=0):
    """
    Post at a fixed depth delta_fixed on every venue simultaneously.
    Total fill rate = sum of individual venue rates.
    Terminal residual at T costs b per unit.
    """
    lams      = [A * np.exp(-k * delta_fixed) for A, k in zip(A_list, k_list)]
    total_lam = sum(lams)
    rng = np.random.default_rng(seed)
    proceeds = np.zeros(n_sims)
    for s in range(n_sims):
        q, tau, total = Q, T, 0.0
        while q > 0 and tau > 1e-9:
            wait = rng.exponential(1.0 / total_lam) if total_lam > 1e-12 else np.inf
            if wait < tau:
                tau -= wait; total += delta_fixed; q -= 1
            else:
                tau = 0
        if q > 0:
            total -= b * q
        proceeds[s] = total
    return proceeds

# Parameters
A1, k1 = 2.0, 0.5   # liquid venue
A2, k2 = 0.5, 1.5   # illiquid venue
b_sor, Q_sor, T_sor, delta_sor = 2.0, 15, 10.0, 1.0

proc_v1  = simulate_sor_fixed([A1],      [k1],      delta_sor, b_sor, Q_sor, T_sor, seed=1)
proc_v2  = simulate_sor_fixed([A2],      [k2],      delta_sor, b_sor, Q_sor, T_sor, seed=2)
proc_sor = simulate_sor_fixed([A1, A2],  [k1, k2],  delta_sor, b_sor, Q_sor, T_sor, seed=3)

print('Fixed-depth strategy (delta=1 tick), Q=15, T=10 min')
print(f'  Venue 1 only (A={A1}, k={k1}): mean={proc_v1.mean():.1f}, std={proc_v1.std():.1f}')
print(f'  Venue 2 only (A={A2}, k={k2}): mean={proc_v2.mean():.1f}, std={proc_v2.std():.1f}')
lam_tot = A1*np.exp(-k1*delta_sor) + A2*np.exp(-k2*delta_sor)
print(f'  Two-venue SOR (lambda_tot={lam_tot:.2f}/min): mean={proc_sor.mean():.1f}, std={proc_sor.std():.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: fill rate curves
d_plt = np.linspace(0, 5, 200)
axes[0].plot(d_plt, A1 * np.exp(-k1 * d_plt), lw=2.5, color='#1565C0',
             label=f'Venue 1 (liquid,  $A={A1}$, $k={k1}$)')
axes[0].plot(d_plt, A2 * np.exp(-k2 * d_plt), lw=2.5, color='#C62828',
             label=f'Venue 2 (illiquid, $A={A2}$, $k={k2}$)')
axes[0].axvline(delta_sor, color='gray', lw=1.5, linestyle=':',
                label=f'$\\delta = {delta_sor}$ (strategy)')
axes[0].set_xlabel('Depth $\\delta$ (ticks)')
axes[0].set_ylabel('Fill rate (fills/min)')
axes[0].set_title('Venue fill rates')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Right: P&L distributions
lo = min(proc_v1.min(), proc_v2.min(), proc_sor.min()) - 1
hi = max(proc_v1.max(), proc_v2.max(), proc_sor.max()) + 1
bins = np.linspace(lo, hi, 60)
for proc, lbl, col in [
    (proc_v1,  'Venue 1 only',    '#1565C0'),
    (proc_v2,  'Venue 2 only',    '#C62828'),
    (proc_sor, 'Two-venue SOR',   '#2E7D32'),
]:
    axes[1].hist(proc, bins=bins, alpha=0.40, color=col, label=lbl, density=True)
    axes[1].axvline(proc.mean(), color=col, lw=2.0, linestyle='--')

axes[1].set_xlabel(f'Total proceeds (ticks, $Q={Q_sor}$, $\\delta={delta_sor}$)')
axes[1].set_ylabel('Density')
axes[1].set_title('Execution P&L distributions (10,000 simulations)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(f'{FIGURES_DIR}/tact_sor.png', dpi=150, bbox_inches='tight')
plt.show()

## 4  Reinforcement learning for execution tactics

Execution tactics can be framed as a Markov Decision Process:

| MDP element | Execution interpretation |
|-------------|--------------------------|
| State $s = (t, q)$ | Elapsed time, remaining inventory |
| Action $a = \delta$ | Placement depth (or market order) |
| Reward | Price improvement $\delta$ per fill; $-b\,q$ at terminal time |
| Transition | Poisson fill event; time step $\Delta t$ |

**Q-learning** maintains a table $Q(s,a)$ and updates on each transition:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha\bigl[r + \gamma\,\max_{a'} Q(s',a') - Q(s,a)\bigr].
$$

The $\varepsilon$-greedy policy starts fully exploratory and decays toward the greedy policy,
recovering the HJB-optimal solution as episode count grows.


In [ ]:
# MDP parameters
A_rl, k_rl, b_rl, T_rl, Q_rl = 1.0, 1.0, 2.0, 10.0, 10
DT_RL   = T_rl / 20          # 0.5-min time steps per episode tick
ACTIONS = np.array([-1, 0, 1, 2, 3, 4])   # -1 = market order (proceeds = 0)

N_T_BINS = 10
N_Q_BINS = Q_rl + 1

def rl_step(tau, q, action_idx, rng):
    d = int(ACTIONS[action_idx])
    if d < 0 or tau <= DT_RL + 1e-9:
        new_q, new_tau = q - 1, max(tau - DT_RL, 0.0)
        done   = new_tau <= 0 or new_q == 0
        reward = -b_rl * new_q if (done and new_q > 0) else 0.0
        return new_tau, new_q, reward, done
    lam    = A_rl * np.exp(-k_rl * d)
    filled = rng.random() < (1 - np.exp(-lam * DT_RL))
    new_q  = (q - 1) if filled else q
    new_tau = tau - DT_RL
    done   = new_tau <= 0 or new_q == 0
    reward = (float(d) if filled else 0.0) + (-b_rl * new_q if (done and new_q > 0) else 0.0)
    return new_tau, new_q, reward, done

# Q-learning
Q_table = np.zeros((N_T_BINS, N_Q_BINS, len(ACTIONS)))
alpha, gamma_rl = 0.12, 0.99
eps_start, eps_end, N_EP = 1.0, 0.05, 6000
eps_decay = (eps_start - eps_end) / N_EP
rng_rl    = np.random.default_rng(42)
ep_rewards = []

for ep in range(N_EP):
    eps  = max(eps_end, eps_start - ep * eps_decay)
    tau, q, total_r = T_rl, Q_rl, 0.0
    for _ in range(120):
        if q == 0 or tau <= 0: break
        tb = min(int(tau / T_rl * N_T_BINS), N_T_BINS - 1)
        qb = min(q, N_Q_BINS - 1)
        a  = (rng_rl.integers(len(ACTIONS)) if rng_rl.random() < eps
              else int(np.argmax(Q_table[tb, qb])))
        ntau, nq, r, done = rl_step(tau, q, a, rng_rl)
        total_r += r
        ntb = min(int(ntau / T_rl * N_T_BINS), N_T_BINS - 1)
        nqb = min(nq, N_Q_BINS - 1)
        target = r if done else r + gamma_rl * np.max(Q_table[ntb, nqb])
        Q_table[tb, qb, a] += alpha * (target - Q_table[tb, qb, a])
        tau, q = ntau, nq
        if done: break
    ep_rewards.append(total_r)

print(f'Training done. Final 500-ep average reward: {np.mean(ep_rewards[-500:]):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: learning curve
window = 300
smooth = np.convolve(ep_rewards, np.ones(window)/window, mode='valid')
ep_ax  = np.arange(len(smooth)) + window // 2
axes[0].scatter(np.arange(N_EP)[::5], np.array(ep_rewards)[::5],
                s=3, alpha=0.15, color='#90CAF9')
axes[0].plot(ep_ax, smooth, lw=2, color='#1565C0', label=f'{window}-ep rolling avg')
axes[0].axhline(np.mean(ep_rewards[-500:]), color='#C62828', lw=1.5,
                linestyle='--', label=f'Converged ≈ {np.mean(ep_rewards[-500:]):.1f}')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total episodic reward')
axes[0].set_title('Q-learning training curve')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Right: greedy policy — best action index mapped to delta value
best_a = np.argmax(Q_table, axis=2)           # (N_T_BINS, N_Q_BINS)
best_d = ACTIONS[best_a]                       # (N_T_BINS, N_Q_BINS)
# flip tau axis: column 0 = start, column -1 = terminal
plot_d = np.flip(best_d[1:, 1:].T, axis=1)    # (Q, T_bins-1)
im = axes[1].imshow(plot_d, aspect='auto', origin='lower',
                    extent=[0, 1, 1/Q_rl, 1],
                    cmap='RdYlGn_r', vmin=-1.5, vmax=4.5)
plt.colorbar(im, ax=axes[1], label='Greedy action $\\delta^*$ (ticks)')
axes[1].set_xlabel('Time elapsed $t/T$')
axes[1].set_ylabel('Remaining fraction $q/Q$')
axes[1].set_title('Learned greedy policy  ($b=2$, $A=1$, $k=1$)')

plt.tight_layout()
fig.savefig(f'{FIGURES_DIR}/tact_rl_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

# Compare RL policy vs HJB reference
print('\nGreedy RL policy at start (t=0):')
tb0 = N_T_BINS - 1   # most time remaining
for q_test in [1, 3, 5, 10]:
    qb  = min(q_test, N_Q_BINS - 1)
    a_rl = int(np.argmax(Q_table[tb0, qb]))
    print(f'  q={q_test:2d}: delta_RL = {ACTIONS[a_rl]}')

## Exercises

**Exercise 1 (Fill probability calibration):** Given a dataset of limit orders with posted depth
$\delta_i$ and fill indicator $y_i \in \{0,1\}$ over a fixed window $\tau$, write the log-likelihood
function for $(A, k)$ and derive the MLE gradient updates.

**Exercise 2 (Analytical ODE for $q=1$):** For a single venue, the HJB ODE for $q=1$ reduces
to an autonomous first-order equation in $\tau$ (since $\Delta H = H(\tau,1) - H(\tau,0) = H(\tau,1)$).
Show that the solution is $H(\tau,1) = \frac{1}{k}\ln(e^{-kb} + A\tau/e)$, and verify
numerically against `compute_hjb`.

**Exercise 3 (Urgency threshold):** Derive the critical inventory level $q^*$ below which $\delta^* > 0$
(limit order is optimal) for a given remaining time $\tau$. How does $q^*$ change as $\tau \to 0$?

**Exercise 4 (Multi-venue independence):** In the SOR model with $K$ venues, show that the
optimal depth for venue $k$ is $\delta^{k,*} = 1/k_k + \Delta H$, where $\Delta H$ is venue-independent.
What happens to the aggressiveness matrix shape as $K$ increases?

**Exercise 5 (Dark pool routing):** Extend `simulate_sor_fixed` to include a dark pool (no
price impact, fill probability determined by latent volume) and show the P&L improvement
for a large order $Q = 50$ units.

**Exercise 6 (DQN extension):** Replace the Q-table in the RL section with a two-layer neural
network using PyTorch. Train on the same MDP and compare the greedy policy to the HJB solution.
